# Oméga — test Colab avec retour automatique vers GitHub

Ce notebook teste le runtime Colab (Python, GPU éventuel, calcul déterministe), puis publie un rapport horodaté dans le dépôt `Abrasax63/Omega-pub`, branche `colab-bridge-2026-11-06`.

## Configuration obligatoire avant « Exécuter tout »

1. Créer un **fine-grained personal access token** GitHub limité au seul dépôt `Omega-pub`.
2. Lui accorder uniquement **Contents: Read and write** (Metadata en lecture seule est automatique). Ne pas lui donner d'autres permissions.
3. Dans Colab, ouvrir le panneau **Secrets** (icône clé à gauche), ajouter le secret nommé exactement `OMEGA_GITHUB_TOKEN`, coller le token dans sa valeur et autoriser ce notebook à y accéder.
4. Ne jamais coller le token dans une cellule, dans le chat ou dans un fichier du dépôt.
5. Exécuter toutes les cellules.

Le token reste dans les Secrets Colab et n'est pas écrit dans le rapport. Le notebook publie un nouveau fichier horodaté sous `results/colab/`. Il ne modifie ni `main` ni `Untitled3.ipynb`.

Si tu ne souhaites pas créer de token, exécute seulement les cellules de test : le rapport sera aussi enregistré dans `/content/omega_colab_smoke_test.json`, mais la publication GitHub sera ignorée.

In [ ]:
import sys, os, json, platform, subprocess
from datetime import datetime, timezone
print('Python :', sys.version.split()[0])
print('Plateforme :', platform.platform())
print('Répertoire courant :', os.getcwd())
gpu = {'detected': False, 'name': None, 'probe': 'not_run'}
try:
    p = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    if p.returncode == 0 and p.stdout.strip():
        gpu.update({'detected': True, 'name': p.stdout.strip().splitlines()[0], 'probe': 'nvidia-smi'})
    else:
        gpu['probe'] = 'nvidia-smi_unavailable_or_no_gpu'
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    gpu['probe'] = type(exc).__name__
try:
    import torch
    gpu['pytorch_available'] = True
    gpu['cuda_available'] = bool(torch.cuda.is_available())
    if gpu['cuda_available']:
        gpu['torch_device_name'] = torch.cuda.get_device_name(0)
except ImportError:
    gpu['pytorch_available'] = False
    gpu['cuda_available'] = False
print(json.dumps(gpu, indent=2, ensure_ascii=False))

In [ ]:
n = 100_000
total = sum(range(n))
expected = n * (n - 1) // 2
assert total == expected == 4_999_950_000
print('TEST EXACT : RÉUSSI')
print('Somme calculée :', total)
print('Valeur attendue :', expected)

In [ ]:
run_time = datetime.now(timezone.utc)
result = {
    'test': 'omega_colab_runtime_smoke_test',
    'timestamp_utc': run_time.isoformat(),
    'python_version': sys.version,
    'platform': platform.platform(),
    'gpu': gpu,
    'arithmetic_test': {'status': 'PASS', 'n': n, 'computed': total, 'expected': expected}
}
local_path = '/content/omega_colab_smoke_test.json'
with open(local_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print('Rapport local créé :', local_path)
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# Publication du rapport sur GitHub via un secret Colab — aucun token affiché.
import base64, requests
from google.colab import userdata

try:
    token = userdata.get('OMEGA_GITHUB_TOKEN')
except Exception:
    token = None

if not token:
    print('PUBLICATION NON EFFECTUÉE : secret OMEGA_GITHUB_TOKEN absent ou non autorisé.')
    print('Le rapport local reste disponible :', local_path)
else:
    stamp = run_time.strftime('%Y%m%dT%H%M%SZ')
    path = f'results/colab/omega_colab_smoke_test_{stamp}.json'
    url = f'https://api.github.com/repos/Abrasax63/Omega-pub/contents/{path}'
    headers = {
        'Authorization': f'Bearer {token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28'
    }
    payload = {
        'message': f'Add Colab runtime report {stamp}',
        'content': base64.b64encode(json.dumps(result, indent=2, ensure_ascii=False).encode('utf-8')).decode('ascii'),
        'branch': 'colab-bridge-2026-11-06'
    }
    response = requests.put(url, headers=headers, json=payload, timeout=30)
    if response.status_code not in (200, 201):
        raise RuntimeError(f'Publication GitHub échouée (HTTP {response.status_code}) : {response.text[:500]}')
    print('PUBLICATION GITHUB : RÉUSSIE')
    print('Fichier :', path)
    print('Branche : colab-bridge-2026-11-06')
    print('Commit :', response.json().get('commit', {}).get('sha', 'non communiqué'))
    del token, headers, payload